# Paramétrage de l'environnement de travail et import des packages

In [1]:
import sys
from pathlib import Path

In [2]:
ROOT = Path.cwd().parents[0]

RAW_DATA = ROOT / "01_data" / "01_raw"
PROCESSED_DATA = ROOT / "01_data" / "02_processed"

%load_ext autoreload
%autoreload 2
sys.path.append(str(ROOT / "03_fonctions"))

In [3]:
import pandas as pd
from fonctions_perso.geo_localisation import calcul_distances

# La base de données

**Import de la base de données**

In [4]:
data_fraud = pd.read_parquet(RAW_DATA / "fraud_data.parquet").astype({
   "cc_num":"object",
   "zip":"object",
   "trans_date_trans_time":"datetime64[ns]",
   "dob":"datetime64[ns]"
})

**Renommage des variables**

In [5]:
data_fraud = data_fraud.rename(columns={
    "trans_date_trans_time":"date_heure_transaction",
    "cc_num":"numero_carte",
    "merchant":"nom_magasin",
    "category":"type_magasin",
    "amt":"montant_transaction",
    "first":"prenom",
    "last":"nom",
    "gender":"sexe",
    "street":"adresse_client",
    "city":"ville_client",
    "state":"etat_client",
    "zip":"code_postal_client",
    "lat":"latitude_domicile_client",
    "long":"longitude_domicile_client",
    "trans_num":"numero_transaction",
    "city_pop":"population_ville_client",
    "job":"profession_client",
    "dob":"date_naissance_client",
    "unix_time":"timestamp_unix_transacation",
    "merch_lat":"latitude_magasin",
    "merch_long":"longitude_magasin",
    "is_fraud":"target"
})

**Nettoyage de la variable "nom_magasin"**

In [6]:
data_fraud["nom_magasin"] = data_fraud["nom_magasin"].str.slice_replace(0,6,"")

**Informations globales**

In [7]:
data_fraud.info() # Pas de valeurs manquantes !

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1852394 entries, 0 to 1852393
Data columns (total 22 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   date_heure_transaction       datetime64[ns]
 1   numero_carte                 object        
 2   nom_magasin                  object        
 3   type_magasin                 object        
 4   montant_transaction          float64       
 5   prenom                       object        
 6   nom                          object        
 7   sexe                         object        
 8   adresse_client               object        
 9   ville_client                 object        
 10  etat_client                  object        
 11  code_postal_client           object        
 12  latitude_domicile_client     float64       
 13  longitude_domicile_client    float64       
 14  population_ville_client      int64         
 15  profession_client            object        
 16  

**Statistiques descriptives**

In [8]:
data_fraud[["montant_transaction","population_ville_client","target"]].describe()

,montant_transaction,population_ville_client,target
count,1.852394e+06,1.852394e+06,1.852394e+06
mean,7.006357e+01,8.864367e+04,5.210015e-03
std,1.592540e+02,3.014876e+05,7.199217e-02
min,1.000000e+00,2.300000e+01,0.000000e+00
25%,9.640000e+00,7.410000e+02,0.000000e+00
50%,4.745000e+01,2.443000e+03,0.000000e+00
75%,8.310000e+01,2.032800e+04,0.000000e+00
max,2.894890e+04,2.906700e+06,1.000000e+00


# Feature engineering & Feature selection

**Création de nouvelles variables**

In [9]:
# Mois de la transaction
data_fraud["mois_transaction"] = data_fraud["date_heure_transaction"].dt.month_name()

# Jour de la transaction
data_fraud["jour_transaction"] = data_fraud["date_heure_transaction"].dt.day_name()

# Heure de la transaction
data_fraud["heure_transaction"] = data_fraud["date_heure_transaction"].dt.hour

# Distance "domicile" - "magasin"
calcul_distances(
    data_fraud,
    "latitude_domicile_client","longitude_domicile_client",
    "latitude_magasin","longitude_magasin",
    unit="km",
    out_col="distance_domicile_magasin"
)

# Age client
data_fraud["age_client"] = data_fraud["date_heure_transaction"].dt.year-data_fraud["date_naissance_client"].dt.year

**Suppression des variables non utilisables**

In [10]:
variables_a_supprimer = ["numero_carte","prenom","nom","sexe","adresse_client","code_postal_client","timestamp_unix_transacation"]

data_fraud = data_fraud.drop(variables_a_supprimer, axis=1)

# Les variables suivantes ne seront pas utilisés ni pour l'EDA ni pour le modèle, elles sont donc supprimées

**Aperçu rapide des données**

In [11]:
data_fraud.head()

,date_heure_transaction,nom_magasin,type_magasin,montant_transaction,ville_client,etat_client,latitude_domicile_client,longitude_domicile_client,population_ville_client,profession_client,date_naissance_client,numero_transaction,latitude_magasin,longitude_magasin,target,mois_transaction,jour_transaction,heure_transaction,distance_domicile_magasin,age_client
0,2019-01-01 00:00:18,"Rippin, Kub and Mann",misc_net,4.97,Moravian Falls,NC,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,36.011293,-82.048315,0,January,Tuesday,0,78.597677,31
1,2019-01-01 00:00:44,"Heller, Gutmann and Zieme",grocery_pos,107.23,Orient,WA,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,49.159047,-118.186462,0,January,Tuesday,0,30.212217,41
2,2019-01-01 00:00:51,Lind-Buckridge,entertainment,220.11,Malad City,ID,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,43.150704,-112.154481,0,January,Tuesday,0,108.206232,57
3,2019-01-01 00:01:16,"Kutch, Hermiston and Farrell",gas_transport,45.00,Boulder,MT,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,47.034331,-112.561071,0,January,Tuesday,0,95.673363,52
4,2019-01-01 00:03:06,Keeling-Crist,misc_pos,41.96,Doe Hill,VA,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,38.674999,-78.632459,0,January,Tuesday,0,77.556851,33
